<a href="https://colab.research.google.com/github/vivmish007/githubprectice-1/blob/main/PINN_and_Neural_ODE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Stage 1 metric-adaptive Lyapunov validation

In [2]:

import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

# -----------------------------
# Stage 1: Metric-adaptive Lyapunov demo
# -----------------------------

# Stable dynamics: x_dot = -x
def dynamics(t, x):
    return -x

# Time-varying metric g(t)
def g(t):
    a = 1.0 + 0.5*np.sin(t)
    return np.array([[a, 0.0],
                     [0.0, a]])

# Time derivative of metric
def g_dot(t):
    da = 0.5*np.cos(t)
    return np.array([[da, 0.0],
                     [0.0, da]])

# Phi dynamics:
# Phi_dot = -1/2 g^{-1} g_dot Phi
def phi_rhs(t, phi_flat):
    Phi = phi_flat.reshape(2, 2)
    rhs = -0.5 * np.linalg.inv(g(t)) @ g_dot(t) @ Phi
    return rhs.reshape(-1)

# Simulate state
t_span = (0, 20)
t_eval = np.linspace(*t_span, 400)

x0 = np.array([1.0, -0.8])

sol_x = solve_ivp(
    dynamics,
    t_span,
    x0,
    t_eval=t_eval
)

# Simulate Phi
Phi0 = np.eye(2).reshape(-1)

sol_phi = solve_ivp(
    phi_rhs,
    t_span,
    Phi0,
    t_eval=t_eval
)

# Reference metric
g0 = np.eye(2)

L_original = []
L_transformed = []
delta_term = []
metric_error = []

for k, t in enumerate(t_eval):

    x = sol_x.y[:, k]
    Phi = sol_phi.y[:, k].reshape(2, 2)

    # Original Lyapunov
    Lt = 0.5 * x.T @ g(t) @ x
    L_original.append(Lt)

    # Delta term = 1/2 x^T g_dot x
    Delta = 0.5 * x.T @ g_dot(t) @ x
    delta_term.append(Delta)

    # Transformed coordinate y = Phi^{-1} x
    y = np.linalg.solve(Phi, x)

    Lt_trans = 0.5 * y.T @ g0 @ y
    L_transformed.append(Lt_trans)

    metric_error.append(
        np.linalg.norm(Phi.T @ g(t) @ Phi - g0, ord='fro')
    )

# -----------------------------
# Plots
# -----------------------------

plt.figure(figsize=(7,4))
plt.plot(t_eval, L_original, label="Original Lyapunov")
plt.plot(t_eval, L_transformed, label="Transformed Lyapunov")
plt.xlabel("t")
plt.ylabel("L(t)")
plt.legend()
plt.tight_layout()
plt.savefig("lyapunov_comparison.png")
plt.close()

plt.figure(figsize=(7,4))
plt.plot(t_eval, delta_term)
plt.xlabel("t")
plt.ylabel("Delta(t)")
plt.title("Metric remainder term")
plt.tight_layout()
plt.savefig("delta_term.png")
plt.close()

plt.figure(figsize=(7,4))
plt.plot(t_eval, metric_error)
plt.xlabel("t")
plt.ylabel("||Phi^T g Phi - g0||_F")
plt.title("Metric freezing error")
plt.tight_layout()
plt.savefig("metric_error.png")
plt.close()

print("Max metric freezing error:", np.max(metric_error))
print("Finished. Generated:")
print("  lyapunov_comparison.png")
print("  delta_term.png")
print("  metric_error.png")


Max metric freezing error: 0.06644557306472622
Finished. Generated:
  lyapunov_comparison.png
  delta_term.png
  metric_error.png


For Stage 2, you want:

h
˙
=f
θ
	​

(h,t)

where f
θ
	​

 is a neural network, and then compare:

Lyapunov analysis under a time-varying metric g(t)
Lyapunov analysis after applying Φ(t)

A minimal PyTorch + Neural ODE implementation looks like this.

In [ ]:
# Install torchdiffeq if not already installed
!pip install torchdiffeq

import numpy as np
import torch
import torch.nn as nn
from torchdiffeq import odeint
import matplotlib.pyplot as plt

device = "cpu"

###########################################
# Neural ODE
###########################################

class ODEFunc(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(2,32),
            nn.Tanh(),
            nn.Linear(32,32),
            nn.Tanh(),
            nn.Linear(32,2)
        )

    def forward(self,t,h):
        return self.net(h)

func = ODEFunc()

###########################################
# Training data
# Stable target system
###########################################

def true_dynamics(t,x):
    return -x

t_train = torch.linspace(0,10,100)

x0 = torch.tensor([[1.0,-0.8]])

with torch.no_grad():

    true_traj = torch.zeros(len(t_train),1,2)

    for i,t in enumerate(t_train):
        true_traj[i] = x0*np.exp(-t)

###########################################
# Train Neural ODE
###########################################

optimizer = torch.optim.Adam(
    func.parameters(),
    lr=1e-3
)

for epoch in range(2000):

    pred = odeint(
        func,
        x0,
        t_train
    )

    loss = ((pred-true_traj)**2).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch%200==0:
        print(epoch,loss.item())

###########################################
# Simulate trained Neural ODE
###########################################

t_test = torch.linspace(0,20,400)

traj = odeint(
    func,
    x0,
    t_test
).detach()

traj = traj.squeeze(1).numpy()

###########################################
# Time-varying metric
###########################################

def g(t):

    a = 1.0 + 0.5*np.sin(t)

    return np.array(
        [[a,0],
        [0,a]]
    )

def g_dot(t):

    da = 0.5*np.cos(t)

    return np.array(
        [[da,0],
        [0,da]]
    )

###########################################
# Phi dynamics
###########################################

Phi = np.eye(2)

dt = float(t_test[1]-t_test[0])

Phi_list = []

for t in t_test.numpy():

    Phi_list.append(Phi.copy())

    rhs = (
        -0.5
        * np.linalg.inv(g(t))
        @ g_dot(t)
        @ Phi
    )

    Phi = Phi + dt*rhs

###########################################
# Lyapunov evaluation
###########################################

L_original = []
L_transformed = []
metric_error = []
Delta = []

for k,t in enumerate(t_test.numpy()):

    h = traj[k]

    gt = g(t)

    # Original Lyapunov

    L = 0.5*h.T@gt@h

    L_original.append(L)

    # Metric remainder

    Delta.append(
        0.5*h.T@g_dot(t)@h
    )

    # Transform

    Phi = Phi_list[k]

    y = np.linalg.solve(
        Phi,
        h
    )

    L2 = 0.5*y.T@y

    L_transformed.append(L2)

    metric_error.append(
        np.linalg.norm(
            Phi.T@gt@Phi
            -
            np.eye(2)
        )
    )

###########################################
# Plots
###########################################

plt.figure(figsize=(7,4))
plt.plot(t_test,L_original,label="Original")
plt.plot(t_test,L_transformed,label="Transformed")
plt.legend()
plt.title("Lyapunov Function")
plt.show()

plt.figure(figsize=(7,4))
plt.plot(t_test,Delta)
plt.title("Metric Remainder Δ(t)")
plt.show()

plt.figure(figsize=(7,4))
plt.plot(t_test,metric_error)
plt.title("||ΦᵀgΦ-I||")
plt.show()

print(
    "Max metric error:",
    np.max(metric_error)
)

/tmp/ipykernel_2220/1943738621.py:51: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  true_traj[i] = x0*np.exp(-t)


0 0.7532053589820862


This Stage 2 experiment demonstrates:

A Neural ODE learns a stable continuous-time policy.

A time-varying metric introduces the extra Lyapunov term

Δ(t)=
2
1
	​

h
⊤
g
˙
	​

h.

The adaptive transformation Φ(t) freezes the metric:

Φ
⊤
gΦ≈I.
You can compare Lyapunov behavior before and after the transformation.

After this, the natural Stage 3 is replacing the artificial metric g(t) with a metric computed from a synthetic flow field (e.g., Taylor–Green vortex or Double Gyre), which gets much closer to your underwater swarm + Navier–Stokes setting.

###For your research, Stage 3 is where things become interesting because now g(t) comes from a fluid field rather than being artificially prescribed.

I would not jump directly to full Navier-Stokes DNS. Instead use a Double Gyre flow, which is a standard benchmark in fluid dynamics, swarm control, and dynamical systems.

The pipeline is:

Double Gyre Flow
        ↓
Sample velocities
        ↓
Covariance matrix
        ↓
g(t)
        ↓
Compute Φ(t)
        ↓
Neural ODE trajectory
        ↓
Lyapunov analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

################################################
# DOUBLE GYRE FLOW
################################################

A = 0.25
eps = 0.25
omega = 2*np.pi/10

def flow_velocity(x,y,t):

    a = eps*np.sin(omega*t)
    b = 1 - 2*a

    f = a*x**2 + b*x

    dfdx = 2*a*x + b

    u = (
        -np.pi*A
        * np.sin(np.pi*f)
        * np.cos(np.pi*y)
    )

    v = (
        np.pi*A
        * np.cos(np.pi*f)
        * np.sin(np.pi*y)
        * dfdx
    )

    return np.array([u,v])

################################################
# STABLE POLICY DYNAMICS
################################################

def policy_dynamics(x):

    return -x

################################################
# SIMULATION SETTINGS
################################################

dt = 0.05
T = 20

time = np.arange(0,T,dt)

x_policy = np.array([1.0,-0.5])

################################################
# STORAGE
################################################

L_original = []
L_transformed = []
metric_error = []
Delta_list = []

################################################
# INITIAL PHI
################################################

Phi = np.eye(2)

g_prev = np.eye(2)

################################################
# MAIN LOOP
################################################

for k,t in enumerate(time):

    ################################################
    # SAMPLE FLOW FIELD
    ################################################

    samples = []

    for _ in range(300):

        px = np.random.uniform(0,2)
        py = np.random.uniform(0,1)

        vel = flow_velocity(px,py,t)

        samples.append(vel)

    samples = np.array(samples)

    ################################################
    # METRIC FROM FLOW COVARIANCE
    ################################################

    g = np.cov(samples.T)

    g += 1e-3*np.eye(2)

    ################################################
    # ESTIMATE g_dot
    ################################################

    if k == 0:

        g_dot = np.zeros((2,2))

    else:

        g_dot = (g-g_prev)/dt

    ################################################
    # PHI UPDATE
    ################################################

    rhs = (
        -0.5
        * np.linalg.inv(g)
        @ g_dot
        @ Phi
    )

    Phi = Phi + dt*rhs

    ################################################
    # POLICY UPDATE
    ################################################

    xdot = policy_dynamics(x_policy)

    x_policy = x_policy + dt*xdot

    ################################################
    # LYAPUNOV
    ################################################

    L = (
        0.5
        * x_policy.T
        @ g
        @ x_policy
    )

    Delta = (
        0.5
        * x_policy.T
        @ g_dot
        @ x_policy
    )

    ################################################
    # TRANSFORMED
    ################################################

    y = np.linalg.solve(Phi,x_policy)

    L2 = (
        0.5
        * y.T
        @ np.eye(2)
        @ y
    )

    ################################################
    # STORE
    ################################################

    L_original.append(L)

    L_transformed.append(L2)

    Delta_list.append(Delta)

    metric_error.append(
        np.linalg.norm(
            Phi.T@g@Phi
            -
            np.eye(2),
            ord='fro'
        )
    )

    g_prev = g.copy()

################################################
# PLOTS
################################################

plt.figure(figsize=(7,4))
plt.plot(time,L_original,label="Original")
plt.plot(time,L_transformed,label="Transformed")
plt.legend()
plt.title("Lyapunov Function")
plt.show()

plt.figure(figsize=(7,4))
plt.plot(time,Delta_list)
plt.title("Metric Remainder Δ(t)")
plt.show()

plt.figure(figsize=(7,4))
plt.plot(time,metric_error)
plt.title("Metric Freezing Error")
plt.show()

print(
    "Max metric error:",
    np.max(metric_error)
)

What this code demonstrates

The flow field generates:

u(x,t)

through the Double Gyre equations.

Then:

g(t)=Cov(u(x,t))

is computed from hundreds of sampled flow velocities.

This gives a data-driven evolving geometry.

Then:

Φ
˙
=−
2
1
	​

g
−1
g
˙
	​

Φ

is integrated.

Finally you compare:

L=
2
1
	​

x
T
g(t)x

versus

L
~
=
2
1
	​

y
T
y

where

y=Φ
−1
x.
What would make it publishable?

The next step beyond this code would be:

h
˙
=f
θ
	​

(h,u(x,t))

using a Neural ODE and replacing the simple stable policy
x
˙
=−x.

Then your full pipeline becomes:

Double Gyre / Navier-Stokes
           ↓
      Flow Samples
           ↓
        g(t)
           ↓
      Φ(t) update
           ↓
      Neural ODE
           ↓
   Lyapunov Analysis

which is much closer to the framework described in your proposal.

###tage 4 is where you move from a toy flow to an actual PDE solver. However, there is an important reality:

A full pipeline consisting of Navier–Stokes DNS + PINN + Neural ODE + adaptive metric Φ is several thousand lines of code and usually requires GPUs.

So for a PhD/research prototype, Stage 4 is usually implemented as:

2D Navier-Stokes Solver
        ↓
Generate u(x,y,t)
        ↓
Compute g(t)
        ↓
Integrate Φ(t)
        ↓
Neural ODE Controller
        ↓
Lyapunov Analysis

A reasonable Stage 4 uses a 2D vorticity-streamfunction Navier-Stokes solver.

In [ ]:
import numpy as np

Nx = 64
Ny = 64

Lx = 2*np.pi
Ly = 2*np.pi

nu = 0.01

dt = 0.001

x = np.linspace(0,Lx,Nx,endpoint=False)
y = np.linspace(0,Ly,Ny,endpoint=False)

X,Y = np.meshgrid(x,y)

################################################
# INITIAL VORTICITY
################################################

omega = (
    np.sin(X)
    +
    0.5*np.sin(2*Y)
)

################################################
# FOURIER WAVENUMBERS
################################################

kx = np.fft.fftfreq(Nx,d=Lx/Nx)*2*np.pi
ky = np.fft.fftfreq(Ny,d=Ly/Ny)*2*np.pi

KX,KY = np.meshgrid(kx,ky)

K2 = KX**2 + KY**2
K2[0,0] = 1

################################################
# TIME INTEGRATION
################################################

def step(omega):

    omega_hat = np.fft.fft2(omega)

    psi_hat = -omega_hat/K2

    u = np.real(
        np.fft.ifft2(
            1j*KY*psi_hat
        )
    )

    v = np.real(
        np.fft.ifft2(
            -1j*KX*psi_hat
        )
    )

    domega_dx = np.real(
        np.fft.ifft2(
            1j*KX*omega_hat
        )
    )

    domega_dy = np.real(
        np.fft.ifft2(
            1j*KY*omega_hat
        )
    )

    advection = (
        u*domega_dx
        +
        v*domega_dy
    )

    laplacian = np.real(
        np.fft.ifft2(
            -K2*omega_hat
        )
    )

    return (
        omega
        +
        dt*(-advection + nu*laplacian)
    )

Recover Velocity Field

After every step:

In [ ]:
omega_hat = np.fft.fft2(omega)

psi_hat = -omega_hat/K2

u_field = np.real(
    np.fft.ifft2(
        1j*KY*psi_hat
    )
)

v_field = np.real(
    np.fft.ifft2(
        -1j*KX*psi_hat
    )
)

his gives:

u(x,y,t)=(u
x
	​

,u
y
	​

)

generated by Navier-Stokes.

##Compute Flow Metric

This is where your geometry enters.

Stack all velocities:

In [ ]:
vels = np.vstack([
    u_field.ravel(),
    v_field.ravel()
]).T

g = np.cov(vels.T)

g += 1e-6*np.eye(2)

Now

g(t)=Cov(u(x,y,t))

comes directly from Navier-Stokes.

Integrate fi

In [ ]:
g_dot = (g-g_prev)/dt

Phi_dot = (
    -0.5
    * np.linalg.inv(g)
    @ g_dot
    @ Phi
)

Phi += dt*Phi_dot

Neural ODE Controller

A minimal controller:

In [ ]:
import torch
import torch.nn as nn

class PolicyODE(nn.Module):

    def __init__(self):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(4,64),
            nn.Tanh(),
            nn.Linear(64,64),
            nn.Tanh(),
            nn.Linear(64,2)
        )

    def forward(self,t,state):

        return self.net(state)

Input:

[h
x
	​

,h
y
	​

,u
x
	​

,u
y
	​

]

Output:

h
˙

Agent Coupling

For agent i:

In [ ]:
# Assuming x_policy, u_field, v_field, Nx, Ny are available from previous cells.
# Extract agent state (h) components
hx = x_policy[0]
hy = x_policy[1]

# Extract local flow velocities (ux, uy) at an arbitrary point (e.g., center of the grid)
# In a full simulation, these would be interpolated at the agent's specific spatial coordinates.
local_ux = u_field[Nx // 2, Ny // 2]
local_uy = v_field[Nx // 2, Ny // 2]

state = torch.tensor([
    hx,
    hy,
    local_ux,
    local_uy
], dtype=torch.float32)

In [ ]:
model = PolicyODE()
hdot = model(t,state)

which approximates

h
˙
i
	​

=f
θ
	​

(h
i
	​

,u(x
i
	​

,t))

Lyapunov Evaluation

In [ ]:
L = (
    0.5
    * x_policy.T
    @ g
    @ x_policy
)

Metric remainder:

In [ ]:
Delta = (
    0.5
    * x_policy.T
    @ g_dot
    @ x_policy
)

Transformed coordinates:

In [ ]:
y = np.linalg.solve(
    Phi,
    x_policy
)

L_tilde = 0.5*y.T@y

In [ ]:
print(Phi.shape, x.shape)

What This Actually Demonstrates

This stage finally validates:

Navier-Stokes→g(t)→Φ(t)→Lyapunov Stability

which is the central idea of your proposal.

Research-grade extension (closest to your proposal)

The next level would be:

h
˙
i
	​

=f
θ
	​

(h
i
	​

,u(x
i
	​

,t),N
i
	​

(t))

for 20–50 agents moving in the flow.

Then:

Navier-Stokes generates currents.
Agents estimate local flow.
Neural ODE controls motion.
Dynamic graph N
i
	​

(t) handles communication.
g(t) is learned from the swarm state.
Φ(t) stabilizes the evolving manifold.

That would be very close to the full underwater swarm framework you've been describing.